In [ ]:
%pip install -Uq "unstructured[all-docs]"

In [ ]:
%pip install -Uq langchain_chroma

In [ ]:
%pip install -Uq langchain-community langchain

In [ ]:
%pip install -Uq python_dotenv

In [ ]:
%pip install langchain_huggingface

In [ ]:
import json
from typing import List

from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from langchain_community.embeddings import SentenceTransformerEmbeddings
from dotenv import load_dotenv

load_dotenv()

Step 1: Partitioning PDF into atomic Elements

In [ ]:
def partition_document(file_path:str) :
    """Extract elements from PDF using unstructured"""
    print(f" Partitioning Document : {file_path}")

    elements = partition_pdf(
        filename = file_path,
        strategy="hi_res", # using hi_res instead of basic_res. It  is slower but accurate
        infer_table_structure=True, # keep tables as structured HTML , not jumbled Text
        extract_image_block_types = ["Image"], # Grab images found in the pdf
        extract_image_block_to_payload = True, # convert image to base64
    )

    print(f"Extracted {len(elements)} of elements")
    return elements

file_path = "./docs/attention-is-all-you-need.pdf"
elements = partition_document(file_path)

In [ ]:
# lets see the different types of elements extracted
set([str(type(el)) for el in elements])

In [ ]:
# to get the contents of the element say we need content of the element 15
elements[15].to_dict()

In [ ]:
elements[14].to_dict()

In [ ]:
elements[1].to_dict()

In [ ]:
# get all the image elements
images = [element for element in elements if element.category == "Image"]
print(f"Found {len(images)} images")

In [ ]:
# tabels
tables = [element for element in elements if element.category == "Table"]
print(f"Found {len(tables)} tables")

In [ ]:
# incase of image of table elements we dont use the "text attribute of the elements but instead
# for images we use raw_text and for tables we use "text-as-html" so we get the actual image and table instead of just text inside them which will be scrambled

Step 2: Chunking elements by Title

In [ ]:
# Now using Unstructred we have paritioned the pdf documents into atomic elements
# we use these atomic elements and group them into chunks. We group them using title

def create_chunks_by_title(elements):
    """create intelligent chunks using title-based strategy"""
    print(f"Creating chunks.....")
    chunks = chunk_by_title(
        elements,
        max_characters=3000, # never exceed 3000 characters per chunk
        new_after_n_chars=2400, # try to start a new chunk after 2400 characters
        combine_text_under_n_chars=500 # Merge tiny chunks under 500 chars with neighbors
    )

    print(f"Created {len(chunks)} chunks")
    return chunks

In [ ]:
chunks = create_chunks_by_title(elements)

In [ ]:
chunks

In [ ]:
# lets get the type of the chunks
set([str(type(chunk)) for chunk in chunks])

In [ ]:
chunks[15].to_dict()

In [ ]:
# to see all the elements inside the chunk 
chunks[4].metadata.orig_elements

In [ ]:
chunks[4].metadata.orig_elements[3].to_dict()

Step 3: summerize the contents of chunks using LLM and vectorize them

In [ ]:
# We are going to summerize th econtents of the chunks , this we are doing because me might have a table so we grab that text as HTML and if there is image we grab base64 of the image
# so first we try to seperate the content types and get their respective format as discussed above

def separate_content_types(chunk):
    """Analyze what types of contents are in the chunk"""
    content_data = {
        'text' : chunk.text,
        'tables' : [],
        'images' : [],
        'types' : ['text']
    }

    # check for tables and images in the original elements
    if hasattr(chunk,'metadata') and hasattr(chunk.metadata,'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__

            # if element is a table extract the data in html format
            if element_type == 'Table':
                content_data['types'].append('table')
                # get html format using getattr(object, attribute, default)
                table_html = getattr(element.metadata,'text_as_html', element.text)
                content_data['tables'].append(table_html)
            elif element_type == "Image":
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)
    content_data['types'] = list(set(content_data['types']))
    return content_data

In [ ]:
# we will use Hugging Face end Point for the LLM model
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from huggingface_hub import InferenceClient
import os
from langchain_core.messages import HumanMessage , SystemMessage

# now for the chunks we got , we create a summary using LLM model, which we use as langchain documents in the vector db
def create_ai_enhanced_summary(text: str, tables: List[str], images : List[str])->str:
    """create AI enhanced summary for mixed content"""
    try:

        hf_token = os.getenv("HF-TOKEN")
        client = InferenceClient(api_key=hf_token)
        
        image_descriptions = []
        
        # STEP 1: Turn images into text descriptions (Very stable on free tier)
        # Using BLIP: a tiny, fast, and always-free model
        for img_b64 in images[:2]: # limit to 2 for speed
            try:
                caption = client.image_to_text(img_b64, model="Salesforce/blip-image-captioning-base")
                image_descriptions.append(caption.generated_text)
            except:
                image_descriptions.append("[Image analysis skipped]")

        # STEP 2: Combine everything into a text prompt
        final_prompt = f"""Summarize this document chunk for a searchable database.
        
        TEXT CONTENT: {text}
        TABLE DATA: {str(tables)}
        IMAGE DESCRIPTIONS: {", ".join(image_descriptions)}
        
        Provide a concise summary of the key facts and topics."""

        # STEP 3: Use a highly-supported free text model
        # Llama 3.1 8B is almost always available on the free serverless tier
        response = client.chat.completions.create(
            model="meta-llama/Llama-3.1-8B-Instruct",
            messages=[{"role": "user", "content": final_prompt}],
            max_tokens=500
        )

        return response.choices[0].message.content

    except Exception as e:
        print(f" AI Summarization Failed : {e}")

        summary = f"{text[:300]}...."
        if tables:
            summary += f"[contains {len(tables)} tables]"
        if images:
            summary += f"[Contains {len(images)} images]"
        return summary

In [ ]:
def summarize_chunks(chunks):
    """Process all the chunks with AI summaries"""
    print("Processign Chunks with AI summaries")

    langchain_documents = []
    total_chunks = len(chunks)

    for i , chunk in enumerate(chunks):
        current_chunk = i+1
        print(f"Processing chunk {current_chunk}/{total_chunks}")

        # analyze chunk content
        content_data = separate_content_types(chunk)

        # debug prints
        print(f"Types found in the chunk : {content_data['types']}")
        print(f"Tables : {len(content_data['tables'])}, images : {len(content_data['images'])}")

        if content_data['tables'] or content_data['images']:
            print(f"Create AI summary for mixed content")
            try:
                enhanced_content = create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'],
                    content_data['images']
                )
                print(f" AI summary created successfully")
                print(f"Enhanced Content preview{enhanced_content[:200]}")
            except Exception as e:
                print(f" AI summary failed using Raw Text")
                enhanced_content = content_data["text"]

        else:
            print(f"Using Raw Text (no tables/images)")
            enhanced_content = content_data["text"]

        doc = Document(
            page_content = enhanced_content,
            metadata = {
                "original_content" : json.dumps({
                    "raw_text":content_data['text'],
                    "tables_html":content_data['tables'],
                    "image_base_64":content_data["images"]
                })
            }
        )

        langchain_documents.append(doc)
    
    print(f"Processed {len(langchain_documents)} chunks")
    return langchain_documents

processed_chunks = summarize_chunks(chunks)

In [ ]:
processed_chunks

In [ ]:
%pip install sentence-transformers

In [ ]:
from langchain_community.embeddings import SentenceTransformerEmbeddings

# with the summarized chunks available create a vector store
def create_vector_store(documents, persist_directory = "dbv1/chroma_db"):
    """create and persist vector db """
    print(f"Creatring the vector store in chromadb")

    embedding_model = SentenceTransformerEmbeddings(model_name="sentence-transformers/all-MiniLM-L12-v2")
    print(f"------ Creating Vector Store------")
    vectorStore = Chroma.from_documents(
        documents = documents,
        embedding = embedding_model,
        persist_directory=persist_directory,
        collection_metadata = {"smsw:space":"cosine"}
    )
    print(f"---Finsihed Creating Vector Store-----")

    return vectorStore

db = create_vector_store(processed_chunks)

In [ ]:
# Lets test the rag
query = "How many attention heads does the transformer use , and what is the dimension of each head?"

retriever = db.as_retriever(search_kwargs = {"k":3})

chunks = retriever.invoke(query)

def generate_final_answer(chunks,query):
    """Generate the final answer using multimodal content"""
    try:
        #initialize the LLM
        # connect to the hugging face
        hf_token = os.getenv("HF-Token")
        client = InferenceClient(api_key = hf_token)

        # Build the text prompt
        prompt_text = f"""Based on the following documents, please answer this question: {query}

CONTENT TO ANALYZE:
"""
        
        for i, chunk in enumerate(chunks):
            prompt_text += f"--- Document {i+1} ---\n"
            
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                
                # Add raw text
                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    prompt_text += f"TEXT:\n{raw_text}\n\n"
                
                # Add tables as HTML
                tables_html = original_data.get("tables_html", [])
                if tables_html:
                    prompt_text += "TABLES:\n"
                    for j, table in enumerate(tables_html):
                        prompt_text += f"Table {j+1}:\n{table}\n\n"
            
            prompt_text += "\n"
        
        prompt_text += """
Please provide a clear, comprehensive answer using the text, tables, and images above. If the documents don't contain sufficient information to answer the question, say "I don't have enough information to answer that question based on the provided documents."

ANSWER:"""

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]
        
        # Add all images from all chunks
        for chunk in chunks:
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                images_base64 = original_data.get("images_base64", [])
                
                for image_base64 in images_base64:
                    message_content.append({
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
                    })
        
        response = client.chat.completions.create(
            model="meta-llama/Llama-3.1-8B-Instruct",
            messages=[{"role": "user", "content": message_content}],
            max_tokens=500
        )

        return response.choices[0].message.content
        
        
    except Exception as e:
        print(f" Answer generation failed: {e}")
        return "Sorry, I encountered an error while generating the answer."

# Usage
final_answer = generate_final_answer(chunks, query)
print(final_answer)